# 🍕 NYC Restaurant Inspections - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

This notebook ingests **NYC restaurant inspection results** into Elasticsearch.

**What this notebook does:**
1. Fetches inspection records straight from [NYC Open Data](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j) - no API key required
2. Normalises and enriches each inspection record (dates, geo point, critical flag)
3. Bulk-indexes everything into your `nyc_restaurant_inspections` index

Run it once to populate the index before building your project.

---
> **Data source:** DOHMH New York City Restaurant Inspection Results, dataset `43nn-pn8j`, served over the public Socrata JSON API. Each row is one violation line from one inspection, so a single restaurant can appear many times - which is exactly what makes it great for aggregations.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import requests
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless

Find your endpoint and API key in:
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)

info = es.info()
print(f'✅ Connected to Elasticsearch {info["version"]["number"]}')

## 2. Fetch NYC restaurant inspection data

Pulls directly from the NYC Open Data Socrata API - no API key needed.

We grab a sample of the most recent inspections. The full dataset is ~250k rows; `SAMPLE_SIZE` keeps the ingest fast for the hack night. **Raise it** (up to a few hundred thousand) if you want the whole city, or add a `$where` clause to focus on one borough or cuisine.

In [ ]:
SAMPLE_SIZE = 50000  # rows to fetch; raise for more coverage, lower for a quicker run

DATA_URL = 'https://data.cityofnewyork.us/resource/43nn-pn8j.json'
params = {
    '$limit': SAMPLE_SIZE,
    '$order': 'inspection_date DESC',
    # Only rows that actually have an inspection date and a restaurant id
    '$where': "inspection_date IS NOT NULL AND camis IS NOT NULL",
}

resp = requests.get(DATA_URL, params=params, timeout=60)
resp.raise_for_status()
rows = resp.json()

print(f'✅ Fetched {len(rows)} inspection rows')
print()
print('Sample row fields:')
for k, v in list(rows[0].items())[:12]:
    print(f'  {k:22s} {v}')

## 3. Create index with explicit mappings

In [ ]:
INDEX = 'nyc_restaurant_inspections'

mapping = {
    'mappings': {
        'properties': {
            # Restaurant identity
            'camis':            {'type': 'keyword'},   # unique restaurant id
            'dba':              {'type': 'keyword'},   # "doing business as" - the name
            'boro':             {'type': 'keyword'},   # MANHATTAN, BROOKLYN, QUEENS, BRONX, STATEN ISLAND
            'building':         {'type': 'keyword'},
            'street':           {'type': 'keyword'},
            'zipcode':          {'type': 'keyword'},
            'cuisine':          {'type': 'keyword'},

            # Inspection
            'inspection_date':  {'type': 'date'},
            'inspection_type':  {'type': 'keyword'},
            'action':           {'type': 'text'},

            # Violation
            'violation_code':        {'type': 'keyword'},
            'violation_description': {'type': 'text'},
            'critical_flag':         {'type': 'keyword'},  # Critical / Not Critical
            'is_critical':           {'type': 'boolean'},

            # Result
            'score':            {'type': 'integer'},   # higher = worse
            'grade':            {'type': 'keyword'},   # A, B, C, ...
            'grade_date':       {'type': 'date'},

            # Location
            'location':         {'type': 'geo_point'},
        }
    }
}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')

es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

## 4. Enrich and ingest all inspections

We clean each row, parse the score to an integer, build a `geo_point`, and add a boolean `is_critical`.
The enriched index can then answer questions like:
- *"What violations did JULIANA'S have on its last inspection?"*
- *"Which borough has the lowest average inspection score?"*
- *"What are the most common critical violations in NYC?"*

In [ ]:
def clean_int(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None

def enrich(r):
    doc = {
        'camis':            r.get('camis'),
        'dba':              (r.get('dba') or '').strip(),
        'boro':             r.get('boro'),
        'building':         r.get('building'),
        'street':           r.get('street'),
        'zipcode':          r.get('zipcode'),
        'cuisine':          r.get('cuisine_description'),
        'inspection_date':  r.get('inspection_date'),
        'inspection_type':  r.get('inspection_type'),
        'action':           r.get('action'),
        'violation_code':        r.get('violation_code'),
        'violation_description': r.get('violation_description'),
        'critical_flag':         r.get('critical_flag'),
        'is_critical':           r.get('critical_flag') == 'Critical',
        'score':            clean_int(r.get('score')),
        'grade':            r.get('grade'),
        'grade_date':       r.get('grade_date'),
    }

    # Build a geo_point when both coordinates are present and valid
    lat, lon = r.get('latitude'), r.get('longitude')
    try:
        lat_f, lon_f = float(lat), float(lon)
        if lat_f != 0.0 and lon_f != 0.0:
            doc['location'] = {'lat': lat_f, 'lon': lon_f}
    except (TypeError, ValueError):
        pass

    # Drop empty values so the index stays clean
    return {k: v for k, v in doc.items() if v not in (None, '')}


docs = [enrich(r) for r in rows]
actions = [{'_index': INDEX, '_source': d} for d in docs]

success, errors = helpers.bulk(es, actions, raise_on_error=False)
es.indices.refresh(index=INDEX)

print(f'✅ Indexed {success} documents ({len(errors)} errors)')
graded = sum(1 for d in docs if 'grade' in d)
critical = sum(1 for d in docs if d.get('is_critical'))
print(f'   Rows with a grade:        {graded}')
print(f'   Critical violation lines: {critical}')

## 5. Verify: quick sense-check queries

Just to confirm the data landed correctly.

In [ ]:
# How many documents? Breakdown by borough and grade.
resp = es.search(index=INDEX, body={
    'size': 0,
    'aggs': {
        'by_boro':  {'terms': {'field': 'boro', 'size': 10}},
        'by_grade': {'terms': {'field': 'grade', 'size': 10}},
    }
})

print(f"Total documents: {resp['hits']['total']['value']}")
print()
print('By borough:')
for b in resp['aggregations']['by_boro']['buckets']:
    print(f"  {b['key']:16s} {b['doc_count']}")
print()
print('By grade:')
for b in resp['aggregations']['by_grade']['buckets']:
    print(f"  {b['key']:4s} {b['doc_count']}")

In [ ]:
# Average inspection score by borough (higher score = worse). Min 50 inspection lines.
resp = es.search(index=INDEX, size=0, body={
    'aggs': {
        'by_boro': {
            'terms': {'field': 'boro', 'size': 10, 'order': {'avg_score': 'desc'}, 'min_doc_count': 50},
            'aggs': {'avg_score': {'avg': {'field': 'score'}}}
        }
    }
})

print('Average inspection score by borough:')
for b in resp['aggregations']['by_boro']['buckets']:
    print(f"  {b['key']:16s} avg score {b['avg_score']['value']:.1f}  ({b['doc_count']} lines)")

In [ ]:
# Most common critical violations - confirms the text field is queryable
resp = es.search(index=INDEX, size=0, body={
    'query': {'term': {'is_critical': True}},
    'aggs': {'top_violations': {'terms': {'field': 'violation_code', 'size': 10}}}
})

print('Most common critical violation codes:')
for b in resp['aggregations']['top_violations']['buckets']:
    print(f"  {b['key']:6s} {b['doc_count']} occurrences")

## 6. Going further with Mistral

The free-text `violation_description` field makes this index a natural fit for Mistral:
- **Semantic search:** back a `semantic_text` field with a Mistral embedding endpoint and search violations by meaning ("pest problems", "unsafe food temperatures") instead of exact codes. See [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md).
- **RAG:** query a restaurant's recent inspections, then ask a Mistral chat model for a plain-English verdict on whether to eat there.
- **Agent:** give a Mistral agent an Elasticsearch search function as a tool, so it can answer "Which cuisine has the worst average score in Queens?" by running the aggregation itself. See [mistral_guide.md](mistral_guide.md).

---
## ✅ Index loaded

Your `nyc_restaurant_inspections` index is ready and verified.

That completes the data ingest. From here, query it however you like - search, aggregations, or a Mistral RAG call. To add Mistral embeddings for semantic search over violation descriptions, see [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md). To call Mistral directly, see [mistral_guide.md](mistral_guide.md).

**To refresh or expand the data**, change `SAMPLE_SIZE` (or the `$where` filter) and re-run cells 2-4. The index is dropped and recreated each time, so there are no duplicates.